# Visualization of segment contributions as a function of solvent ratio (CLI-GAT-SEI)
Put a SMILES string to use the CLI-GAT-SEI model.

In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import SSC
from utils import visualize_ratio_sweep_cli

## CLI-GAT-SEI model

In [2]:
CLI_segment = SSC.src.SSCAnalyzer.SSCAnalyzer_v1p3("CLI_segment_sum_contribution")

founded model: ./Trained_models/CLI_segment_sum_contribution
Device: cuda
#params: 176131
Loading params from ./Trained_models/CLI_segment_sum_contribution/final.pth


/home/ainshin/miniconda3/envs/tlc/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Sculptor Index: (6, 2, 0)


## Put/replace SMILES 

Run by only changing `SMILES`.

In [3]:
# Put/replace SMILES, ratio
SMILES = "C1=CC=C(C=C1)N"          # molecule SMILES
elu1_smiles = "CCCCCC"           # eluent1 SMILES (ex: "CCCCCC")
elu2_smiles = "CCOC(C)=O"        # eluent2 SMILES (ex: "CCOC(C)=O")
ratio1 = [0.5]                   # eluent1 ratio (ex: [1.0, 0.9, ..., 0.0])
ratio2 = [0.5]                   # eluent2 ratio list (ex: [0.0, 0.1, ..., 1.0])
MODE = "ratio_weighted"       # 

if not (SMILES and elu1_smiles and elu2_smiles and len(ratio1) and len(ratio2)):
    raise ValueError("Put/replace SMILES, elu1_smiles, elu2_smiles, ratio1, ratio2.")
if len(ratio1) != len(ratio2) or not np.allclose(np.add(ratio1, ratio2), 1.0):
    raise ValueError("The sum of ratio1 and ratio2 must be 1..")

custom_table, custom_scores = visualize_ratio_sweep_cli(
    CLI_segment, SMILES,
    hex_smiles=elu1_smiles, ea_smiles=elu2_smiles, ea_fractions=list(ratio2),
    label1=elu1_smiles, label2=elu2_smiles, mode=MODE,
)

custom_summary = custom_table.pivot_table(
    index=[elu2_smiles, elu1_smiles, "Rf_pred"], columns=["key", "frag", "frag_smiles"], values="contribution"
)
custom_summary.round(4)

key                       ratio_weighted_contribution1           \
frag                                                 0        1   
frag_smiles                                         *N c1ccccc1   
CCOC(C)=O CCCCCC Rf_pred                                          
0.5       0.5    0.692014                      -0.1955   0.0596   

key                       ratio_weighted_contribution2           
frag                                                 0        1  
frag_smiles                                         *N c1ccccc1  
CCOC(C)=O CCCCCC Rf_pred                                         
0.5       0.5    0.692014                       0.0508   0.8158

## HEX/EA

Run by only changing `SMILES`.

In [4]:
SMILES = "C1=CC=C(C=C1)N"

HEX_SMILES = "CCCCCC"
EA_SMILES = "CCOC(C)=O"
EA_FRACTIONS = np.round(np.linspace(0.0, 1.0, 11), 1)  # 0.0, 0.1, ..., 1.0


sweep_table, sweep_scores = visualize_ratio_sweep_cli(
    CLI_segment, SMILES,
    hex_smiles=HEX_SMILES, ea_smiles=EA_SMILES, ea_fractions=EA_FRACTIONS,
    label1="HEX", label2="EA", mode="ratio_weighted",
)

sweep_summary = sweep_table.pivot_table(
    index=["EA", "HEX", "Rf_pred"], columns=["key", "frag", "frag_smiles"], values="contribution"
)
sweep_summary.round(4)

key              ratio_weighted_contribution1           \
frag                                        0        1   
frag_smiles                                *N c1ccccc1   
EA  HEX Rf_pred                                          
0.0 1.0 0.381389                      -0.3767   0.1188   
0.1 0.9 0.443514                      -0.3422   0.1070   
0.2 0.8 0.505639                      -0.3067   0.0952   
0.3 0.7 0.567764                      -0.2704   0.0834   
0.4 0.6 0.629889                      -0.2333   0.0715   
0.5 0.5 0.692014                      -0.1955   0.0596   
0.6 0.4 0.754139                      -0.1572   0.0477   
0.7 0.3 0.816264                      -0.1183   0.0358   
0.8 0.2 0.878389                      -0.0791   0.0239   
0.9 0.1 0.940514                      -0.0396   0.0119   
1.0 0.0 1.002639                       0.0000   0.0000   

key              ratio_weighted_contribution2           
frag                                        0        1  
frag_smiles                                *N c1ccccc1  
EA  HEX Rf_pred                                         
0.0 1.0 0.381389                       0.0000   0.0000  
0.1 0.9 0.443514                       0.0102   0.2249  
0.2 0.8 0.505639                       0.0203   0.4281  
0.3 0.7 0.567764                       0.0305   0.5957  
0.4 0.6 0.629889                       0.0406   0.7236  
0.5 0.5 0.692014                       0.0508   0.8158  
0.6 0.4 0.754139                       0.0609   0.8793  
0.7 0.3 0.816264                       0.0710   0.9219  
0.8 0.2 0.878389                       0.0811   0.9499  
0.9 0.1 0.940514                       0.0912   0.9680  
1.0 0.0 1.002639                       0.1013   0.9796

## DCM / MeOH

Run by only changing `SMILES`.

In [5]:
SMILES = "C1=CC=C(C=C1)N"
DCM_SMILES = "ClCCl"
MEOH_SMILES = "CO"

dcm_table, dcm_scores = visualize_ratio_sweep_cli(
    CLI_segment, SMILES,
    hex_smiles=DCM_SMILES, ea_smiles=MEOH_SMILES,
    label1="DCM", label2="MeOH", mode="ratio_weighted",
)

dcm_summary = dcm_table.pivot_table(
    index=["MeOH", "DCM", "Rf_pred"], columns=["key", "frag", "frag_smiles"], values="contribution"
)
dcm_summary.round(4)

key               ratio_weighted_contribution1           \
frag                                         0        1   
frag_smiles                                 *N c1ccccc1   
MeOH DCM Rf_pred                                          
0.0  1.0 0.584892                      -0.0807   0.5899   
0.1  0.9 0.622437                      -0.0727   0.5440   
0.2  0.8 0.659982                      -0.0646   0.4945   
0.3  0.7 0.697527                      -0.0566   0.4417   
0.4  0.6 0.735071                      -0.0485   0.3855   
0.5  0.5 0.772616                      -0.0404   0.3264   
0.6  0.4 0.810161                      -0.0324   0.2646   
0.7  0.3 0.847706                      -0.0243   0.2005   
0.8  0.2 0.885251                      -0.0162   0.1347   
0.9  0.1 0.922796                      -0.0081   0.0677   
1.0  0.0 0.960341                       0.0000   0.0000   

key               ratio_weighted_contribution2           
frag                                         0        1  
frag_smiles                                 *N c1ccccc1  
MeOH DCM Rf_pred                                         
0.0  1.0 0.584892                       0.0000   0.0000  
0.1  0.9 0.622437                       0.0844   0.1354  
0.2  0.8 0.659982                       0.1676   0.2659  
0.3  0.7 0.697527                       0.2485   0.3873  
0.4  0.6 0.735071                       0.3261   0.4966  
0.5  0.5 0.772616                       0.3995   0.5922  
0.6  0.4 0.810161                       0.4681   0.6736  
0.7  0.3 0.847706                       0.5315   0.7413  
0.8  0.2 0.885251                       0.5895   0.7967  
0.9  0.1 0.922796                       0.6419   0.8414  
1.0  0.0 0.960341                       0.6890   0.8769